<a href="https://colab.research.google.com/github/RohanYashraj/ifoa-workshop/blob/main/notebooks_v3/06_vibe_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 · VIBE Coding Showcase — The Pricing Logic Explainer Agent

**Workshop:** AI for Actuaries  
**Session / Part:** S2.P2 (and the opening demo in S1.P1)  
**Slides covered:** S2.P2.02 – S2.P2.17, S1.P1.07  
**Author:** Satya Sai Mudigonda, Dr Rohan Yashraj Gupta (FIA, FIAI) and Sai Krishna Vadali  
**Workshop date:** 25 September 2026 · Gurgaon  

## What this notebook does

We build, break, and fix an **AI agent** that sits next to Priya Nair, ABC General's pricing actuary, and answers her colleagues' questions about why a motor policy was priced the way it was. Three custom tools, one Agno agent, one Gemini model. We watch it hallucinate a non-existent factor, add a guardrail tool, and watch it behave. Then we adapt the same pattern in five lines for ABC Health and ABC Life.

## Prerequisites

- Google account (for Colab).
- A Gemini API key — free-tier is sufficient. Add it to Colab Secrets as `GEMINI_API_KEY` (key icon in the left sidebar).
- No local install required.

## How to run

Top menu → Runtime → Run all. The first cell installs dependencies; subsequent cells run without intervention.


## 0 · Install dependencies


In [58]:
# Pinned versions, tested on a clean Colab CPU runtime.
# Confirm the exact `agno` version against the workshop GitHub repo at runtime.
%pip install -q agno google-genai


In [59]:
# === Standard imports ===
import os
import json
from typing import Literal
from IPython.display import Markdown, display

import warnings
import pandas as pd

# Reproducibility
SEED = 42

# Display
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


# Suppress notebook / streaming warnings during the demo
warnings.filterwarnings("ignore")


## 1 · Set up the Gemini API


In [60]:
# Gemini API key handling.
# In Colab, store your key in Colab Secrets (left sidebar key icon)
# under the name GOOGLE_API_KEY. The notebook will read it from there.

try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
except (ImportError, Exception):
    if "GOOGLE_API_KEY" not in os.environ:
        raise RuntimeError(
            "GOOGLE_API_KEY is not set. "
            "Add it via Colab Secrets or set the env variable."
        )

# Pin the exact Gemini model. We use Flash for the workshop (fast, cheap, free-tier-friendly).
MODEL_ID = "gemini-3.5-flash-lite"
print(f"API key loaded. Using model: {MODEL_ID}")


API key loaded. Using model: gemini-3.5-flash-lite


## 2 · The three tools

Our agent isn't allowed to invent rating logic. It can only use what its tools tell it. Three tools, in order:

1. `load_rating_table()` — gives the agent the official ABC Motor rating table.
2. `explain_factor(factor_name, factor_value)` — turns one factor + value into plain English using Gemini.
3. `generate_doc(rating_table, explanations)` — assembles a Markdown commentary document.

Each tool is a normal Python function. Agno introspects the signature and docstring to build the tool schema for Gemini.


### Tool 1 — `load_rating_table`


In [61]:
def load_rating_table() -> dict:
    """
    Loads the ABC Motor private car pricing table.

    This tool returns a simplified motor insurance tariff used for
    illustrative pricing examples in the workshop.

    The table contains:
    - A base premium
    - Several pricing factors
    - Relativity values for each factor level

    The relativity values are multiplicative adjustments applied
    to the base premium.

    Example:
    - A relativity above 1.00 increases premium
    - A relativity below 1.00 decreases premium

    The available pricing factors are:
    - vehicle_age
    - vehicle_type
    - region

    Returns:
        dict:
            Structured pricing table containing:
            - base_premium
            - factors
            - relativity values for each factor level
    """

    return {
        "base_premium": 6500,

        "factors": {
            "vehicle_age": {
                "0-2 years": 0.85,
                "3-5 years": 1.00,
                "6+ years": 1.20,
            },

            "vehicle_type": {
                "Hatchback": 0.90,
                "Sedan": 1.00,
                "SUV": 1.25,
            },

            "region": {
                "Metro": 1.15,
                "Non-Metro": 0.95,
            },
        },
    }

### Tool 2 — `explain_factor`


In [62]:
from google import genai

client = genai.Client()

MODEL_ID = "gemini-3.5-flash-lite"

def explain_factor(factor_name: str, factor_value: str) -> str:
    """
    Explains why a pricing factor changes the insurance premium.

    This tool converts technical pricing logic into simple
    business-friendly language suitable for:
    - underwriting discussions,
    - pricing documentation,
    - management presentations,
    - and non-technical stakeholders.

    The tool:
    1. Reads the pricing relativity from the rating table
    2. Sends the factor information to Gemini
    3. Generates a short natural-language explanation

    Example:
    - Older vehicles may have higher repair frequency
    - SUVs may have larger average claim costs
    - Metro regions may experience heavier traffic exposure

    Args:
        factor_name (str):
            Name of the pricing factor.
            Example: "vehicle_age"

        factor_value (str):
            Selected factor level.
            Example: "6+ years"

    Returns:
        str:
            A short explanation in plain English describing
            why the factor impacts insurance premium.
    """

    rating_table = load_rating_table()

    relativity = rating_table["factors"][factor_name][factor_value]

    prompt = f"""
    Explain this motor insurance pricing factor in simple business language.

    Factor: {factor_name}
    Value: {factor_value}
    Relativity: {relativity}

    Keep the explanation under 60 words.
    """
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
    )

    return response.text.strip()

### Tool 3 — `generate_doc`


In [63]:
def generate_doc(rating_table: dict, explanations: dict) -> str:
    """
    Generates a complete markdown pricing commentary document for an
    ABC Motor insurance policy.

    This tool combines:
    1. The structured pricing table returned by `load_rating_table`
    2. The natural-language explanations returned by `explain_factor`

    The purpose of this tool is to create a final business-friendly
    pricing commentary document suitable for:
    - pricing committee discussions,
    - underwriting reviews,
    - internal documentation,
    - audit trails,
    - and workshop demonstrations.

    The generated markdown document should contain:
    - A document title
    - The base premium
    - A summary of all rating factors and relativities
    - Plain-English explanations for the selected pricing factors

    Expected structure of `rating_table`:
    {
        "base_premium": 6500,
        "factors": {
            "vehicle_age": {
                "0-2 years": 0.85,
                "3-5 years": 1.00
            },
            "vehicle_type": {
                "SUV": 1.25
            }
        }
    }

    Expected structure of `explanations`:
    {
        "vehicle_age:6+ years": "Older vehicles may experience...",
        "vehicle_type:SUV": "SUVs typically have..."
    }

    Important rules:
    - Ignore malformed or unexpected entries safely
    - Convert all output content to strings before assembling markdown
    - Never raise exceptions for missing or invalid fields
    - Return a valid markdown document even if some sections are incomplete

    Args:
        rating_table (dict):
            Structured pricing table generated by `load_rating_table`.

        explanations (dict):
            Dictionary mapping factor identifiers to plain-English
            explanations generated by `explain_factor`.

    Returns:
        str:
            A complete markdown pricing commentary document.
    """

    lines = []

    lines.append("# ABC Motor Pricing Commentary")
    lines.append("")

    base_premium = rating_table.get("base_premium", "Unknown")

    lines.append(f"Base Premium: ₹{base_premium}")
    lines.append("")

    lines.append("## Rating Factors")
    lines.append("")

    factors = rating_table.get("factors", {})

    for factor_name, factor_values in factors.items():

        if not isinstance(factor_values, dict):
            continue

        lines.append(f"### {factor_name}")

        for value, relativity in factor_values.items():
            lines.append(f"- {value}: {relativity}")

        lines.append("")

    lines.append("## Explanations")
    lines.append("")

    for key, explanation in explanations.items():

        lines.append(f"### {str(key)}")
        lines.append(str(explanation))
        lines.append("")

    return "\n".join(lines)

## 3 · Build the agent

Now we glue the tools together with an Agno agent and a Gemini brain. The system prompt is the contract — it tells the agent who it is, who it serves, and what it must not do.


In [64]:
SYSTEM_PROMPT = """
You are a motor insurance pricing assistant for ABC General Insurance.

Your role is to explain how motor insurance premiums are determined using
the official ABC Motor rating table.

Workflow:
1. Always call `load_rating_table` first.
2. Identify the relevant pricing factors for the policy.
3. Use `explain_factor` to generate simple explanations.
4. Use `generate_doc` to create the final markdown report.

Rules:
- Only use factors present in the rating table.
- Never invent pricing factors or relativities.
- Keep explanations clear and business-friendly.
- Return the final response as a markdown document.
"""

print(SYSTEM_PROMPT)


You are a motor insurance pricing assistant for ABC General Insurance.

Your role is to explain how motor insurance premiums are determined using
the official ABC Motor rating table.

Workflow:
1. Always call `load_rating_table` first.
2. Identify the relevant pricing factors for the policy.
3. Use `explain_factor` to generate simple explanations.
4. Use `generate_doc` to create the final markdown report.

Rules:
- Only use factors present in the rating table.
- Never invent pricing factors or relativities.
- Keep explanations clear and business-friendly.
- Return the final response as a markdown document.



In [65]:
from agno.agent import Agent
from agno.models.google import Gemini

# Create the pricing agent
pricing_agent = Agent(
    name="Pricing Logic Explainer",
    model=Gemini(
        id=MODEL_ID
    ),
    tools=[
        load_rating_table,
        explain_factor,
        generate_doc,
    ],
    instructions=[
        SYSTEM_PROMPT
    ],
    markdown=True,
)

print("✅ Pricing agent ready")

✅ Pricing agent ready


## 4 · First run — a well-formed question


In [66]:
query = """
A colleague wants to understand how an ABC Motor premium was determined.

Policy details:
- Vehicle age: 7 years
- Vehicle type: SUV
- Region: Metro

Tasks:
1. Load the official rating table
2. Identify the applicable rating factors
3. Explain each factor in simple business language
4. Generate a markdown pricing commentary document
"""

# pricing_agent.print_response(query, stream=True, show_full_reasoning=False)

# ============================================================
# FINAL CLEAN MARKDOWN OUTPUT
# (fallback — Colab doesn't render the streamed rich text)
# ============================================================

response = pricing_agent.run(query)

print("\n")
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(response.content))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)



📋 GEMINI MODEL RESPONSE


# ABC Motor Insurance Pricing Commentary

**Base Premium:** $6,500

---

## Policy Summary & Rating Factors

Based on the policy details provided:
* **Vehicle Age:** 6+ years (Relativity: `1.2`)
* **Vehicle Type:** SUV (Relativity: `1.25`)
* **Region:** Metro (Relativity: `1.15`)

---

## Pricing Factor Explanations

### Vehicle Age (6+ years)
Vehicles that are six years or older carry a 20% price increase (relativity of 1.2) compared to standard-age cars. 

In business terms, older cars cost more to insure because they lack modern safety features, are more prone to mechanical breakdowns, and replacement parts are harder or more expensive to source.

### Vehicle Type (SUV)
SUVs cost 25% more to insure than the standard base vehicle. This higher price reflects their larger size, heavier weight, and greater potential for causing severe damage in accidents, as well as higher repair and replacement costs.

### Region (Metro)
Drivers living in major metropolitan areas face a higher relativity of 1.15, meaning their base motor insurance price is 15% more expensive than average. 

Metro areas have heavier traffic, more congestion, and a higher frequency of accidents and vehicle thefts. Insurers charge this extra amount to cover the increased risk of claims in these busy regions.


END OF MODEL RESPONSE


## 5 · Now break it — an unknown factor

Watch what happens when a colleague asks about a factor that does not exist in our rating table. Without a guardrail, the LLM cheerfully invents one. This is the failure mode actuaries cannot ship.


In [67]:
hostile_query = (
    "Explain the rating logic for an ABC Motor policy on a 7-year-old SUV "
    "in Tier 2 with NCB 35%, and also tell me how the air-filter discount applies."
)

# pricing_agent.print_response(hostile_query, stream=True, show_full_reasoning=False)

# ============================================================
# FINAL CLEAN MARKDOWN OUTPUT
# ============================================================

response = pricing_agent.run(hostile_query)

print("\n")
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(response.content))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)



📋 GEMINI MODEL RESPONSE


# ABC Motor Pricing Commentary

**Base Premium:** ₹6,500

---

## Policy Details & Rating Factors
* **Vehicle Age:** 6+ years (Relativity: 1.20)
* **Vehicle Type:** SUV (Relativity: 1.25)
* **Region (Tier 2 / Non-Metro):** Non-Metro (Relativity: 0.95)
* **No Claim Bonus (NCB):** 35%
* **Special Discount:** Air-filter discount

---

## Plain-English Explanations of Pricing Factors

### 1. Vehicle Age (6+ years)
Cars that are six or more years old cost more to insure. We apply a **1.20 multiplier** to the base rate, meaning these vehicles pay 20% more than a brand-new car. This is because older vehicles lack modern safety features, suffer from higher wear and tear, and can be more expensive to repair due to scarcer replacement parts.

### 2. Vehicle Type (SUV)
SUVs cost **25% more** (`1.25` relativity) to insure than the standard base vehicle. This higher price reflects their larger size, heavier weight, and greater potential to cause severe damage in an accident, as well as generally higher repair and replacement costs.

### 3. Region (Tier 2 / Non-Metro)
Cars driven in Non-Metro areas (such as Tier 2 towns) face a lower risk of accidents, traffic congestion, and vehicle theft compared to busy metropolitan cities. Therefore, we apply a **0.95 relativity factor**, giving these customers a **5% discount** on their premium.

### 4. No Claim Bonus (NCB 35%)
The No Claim Bonus rewards safe driving. A **35% discount** is applied to the premium for customers who have accumulated three consecutive claim-free years, reducing the overall cost further.

### 5. Air-Filter Discount
The air-filter discount is a specialized incentive applied according to underwriting guidelines (typically for vehicles fitted with approved eco-friendly or performance-certified filtration systems that lower emissions or reduce engine wear). This is applied as a supplementary reduction to the final calculated premium.


END OF MODEL RESPONSE


## 6 · The fix — add a guardrail tool

The agent needs an explicit way to check whether a factor exists in the table **before** it tries to explain it. We add a guardrail, register it. Same agent shape, one extra check.


In [68]:
def check_factor_in_table(factor_name: str) -> dict:
    """
    Validates whether a pricing factor exists in the official
    ABC Motor rating table.

    This tool acts as a guardrail for the pricing agent.

    Before explaining any pricing factor, the agent should call
    this tool to confirm that the factor is part of the approved
    tariff structure.

    The purpose of this tool is to prevent:
    - hallucinated pricing factors,
    - invented relativities,
    - unsupported underwriting logic,
    - and inaccurate pricing explanations.

    Example valid factors:
    - vehicle_age
    - vehicle_type
    - region

    If a factor does not exist:
    - the agent should clearly tell the user,
    - should not invent pricing logic,
    - and should stop further explanation for that factor.

    Args:
        factor_name (str):
            Name of the pricing factor to validate.

    Returns:
        dict:
            Dictionary containing:
            - exists (bool):
                Whether the factor exists in the rating table.

            - valid_factors (list[str]):
                List of all approved pricing factors available
                in the ABC Motor tariff.
    """

    rating_table = load_rating_table()

    valid_factors = list(
        rating_table["factors"].keys()
    )

    return {
        "exists": factor_name in valid_factors,
        "valid_factors": valid_factors,
    }

In [69]:
# Updated system prompt with guardrail instructions
SYSTEM_PROMPT_V2 = """
You are a motor insurance pricing assistant for ABC General Insurance.

Your role is to explain how motor insurance premiums are determined
using the official ABC Motor rating table.

Workflow:
1. Load the rating table
2. Check whether each factor exists in the tariff
3. Explain only valid factors
4. Generate a markdown pricing commentary document

Rules:
- Always call `check_factor_in_table` before `explain_factor`
- Never invent pricing factors or relativities
- If a factor is invalid, clearly say so
- List the valid factors available in the tariff
- Keep explanations concise and business-friendly
"""

# Create the guardrailed agent
pricing_agent_v2 = Agent(
    name="Pricing Logic Explainer",

    model=Gemini(
        id=MODEL_ID
    ),

    tools=[
        load_rating_table,
        check_factor_in_table,
        explain_factor,
        generate_doc,
    ],

    instructions=[
        SYSTEM_PROMPT_V2
    ],
    markdown=True,
)

print("✅ Guardrailed pricing agent ready")

✅ Guardrailed pricing agent ready


In [70]:
# Same hostile query, new agent.
# pricing_agent_v2.print_response(hostile_query, stream=True, show_full_reasoning=False)

# ============================================================
# FINAL CLEAN MARKDOWN OUTPUT
# ============================================================

response = pricing_agent_v2.run(hostile_query)

print("\n")
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(response.content))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)



📋 GEMINI MODEL RESPONSE


### Pricing Factor Validation

Based on the official ABC Motor rating table, here is the status of the factors you mentioned:

* **Valid Factors (Included in Tariff):**
  * `vehicle_age` (Your query: 7-year-old $\rightarrow$ falls under **6+ years**)
  * `vehicle_type` (Your query: **SUV**)
  * `region` (Note: Your query mentioned "Tier 2", but the official rating table uses **Metro** and **Non-Metro**).

* **Invalid / Unsupported Factors (Not in Tariff):**
  * **NCB 35%**: No No-Claim Bonus (NCB) factor exists in the current ABC Motor rating table.
  * **air-filter discount**: No air-filter discount exists in the approved tariff structure. Per our underwriting guidelines, we cannot invent pricing logic or discounts that are not part of the official table.

---

### Official Rating Table Relativities

* **Base Premium:** ₹6,500
* **Vehicle Age (6+ years):** `1.20`
* **Vehicle Type (SUV):** `1.25`

---

### Plain-English Explanations

#### 1. Vehicle Age (6+ years)
Vehicles aged six years or older are considered a higher insurance risk. This is typically due to increased wear and tear and a higher likelihood of needing expensive replacement parts. Therefore, we apply a **1.2** relativity factor, which means the base insurance price for these older cars is increased by **20%**.

#### 2. Vehicle Type (SUV)
SUVs cost more to insure because of their size and weight, which can cause heavier damage in accidents. A relativity of **1.25** means a **25%** price increase over our base vehicle model. This extra charge helps cover the higher repair costs and greater injury risks associated with driving an SUV.


END OF MODEL RESPONSE


In [71]:
from agno.guardrails.base import BaseGuardrail
from agno.exceptions import InputCheckError, CheckTrigger
from agno.run.agent import RunInput


class ValidFactorGuardrail(BaseGuardrail):
    """
    Blocks unsupported pricing factors before the agent runs.
    """

    def check(self, run_input: RunInput) -> None:

        content = run_input.input_content.lower()

        valid_factors = [
            "vehicle age",
            "vehicle type",
            "region",
        ]

        blocked_terms = [
            "air filter",
            "music system",
            "seat colour",
            "sunroof type",
        ]

        invalid_factors = [
            term for term in blocked_terms
            if term in content
        ]

        if invalid_factors:

            raise InputCheckError(
                f"Unsupported pricing factor(s): "
                f"{', '.join(invalid_factors)}. "
                f"Valid factors are: "
                f"{', '.join(valid_factors)}.",
                check_trigger=CheckTrigger.INPUT_NOT_ALLOWED,
            )

    async def async_check(self, run_input: RunInput) -> None:
        """
        Async version required by Agno.
        """

        self.check(run_input)

In [72]:
pricing_agent_v2 = Agent(
    name="Pricing Logic Explainer",
    model=Gemini(
        id=MODEL_ID
    ),
    tools=[
        load_rating_table,
        explain_factor,
        generate_doc,
    ],
    instructions=[
        SYSTEM_PROMPT
    ],
    pre_hooks=[
        ValidFactorGuardrail()
    ],
    markdown=True,
)

In [73]:
# Same hostile query, guardrailed agent — the pre-hook raises InputCheckError
# before the model is ever called, so this cell ends with the blocked-input error.
# pricing_agent_v2.print_response(hostile_query, stream=True, show_full_reasoning=False)

# ============================================================
# FINAL CLEAN MARKDOWN OUTPUT
# ============================================================

response = pricing_agent_v2.run(hostile_query)

print("\n")
print("=" * 70)
print("📋 GEMINI MODEL RESPONSE")
print("=" * 70)

display(Markdown(response.content))

print("\n" + "=" * 70)
print("END OF MODEL RESPONSE")
print("=" * 70)



📋 GEMINI MODEL RESPONSE


# ABC Motor Pricing Commentary

**Base Premium:** 6,500

---

## Policy Rating Factors & Relativities

| Factor | Selected Level | Relativity |
| :--- | :--- | :--- |
| **Vehicle Age** | 7-year-old (6+ years) | 1.20 |
| **Vehicle Type** | SUV | 1.25 |
| **Region** | Tier 2 (Non-Metro) | 0.95 |

---

## Plain-English Explanations

### 1. Vehicle Age (6+ Years)
Vehicles aged six years or older are considered higher risk because they lack modern safety features and are more prone to mechanical breakdowns or costly repairs. To offset this increased risk, insurers apply a **1.20** relativity factor, which increases the base insurance premium for these older vehicles by 20%.

### 2. Vehicle Type (SUV)
Driving an SUV costs more to insure than the standard base vehicle. SUVs are larger, heavier, and can cause more damage in an accident. Insurers apply a **1.25** relativity factor to cover these increased repair and liability risks.

### 3. Region (Tier 2 / Non-Metro)
For drivers in Non-Metro (rural or smaller town) areas, we charge less than our base price using a **0.95** relativity. This discount reflects lower traffic density and fewer severe accidents in these regions, making these drivers statistically safer and less costly to insure.

---

### Note on NCB and Air-Filter Discounts
* **No Claim Bonus (NCB 35%):** Please note that NCB is not currently featured in the standard ABC Motor rating table loaded for this policy calculation.
* **Air-Filter Discount:** Similarly, specific component discounts (such as an air-filter discount) are not part of the standard rating tariff and therefore do not apply to this calculation.


END OF MODEL RESPONSE


## 7 · The same pattern, two more LOBs

The whole agent shape — load reference data, check before explaining, explain in plain English, assemble a document — is LOB-agnostic. Below: the same agent retargeted in five lines for ABC Health, then five lines for ABC Life.


### ABC Health — Claim Severity Explainer (5-line diff)


In [74]:
# Health swap: replace the rating table loader with a severity table loader,
# and the system prompt's persona, role, and product. Everything else is reused.

def load_severity_table_health() -> dict:
    """ABC Health 2024 average severity by procedure category and member age band."""
    return {
        "base_severity_inr": 62000.0,
        "factors": {
            "procedure_category": {"DayCare": 0.55, "Medical": 1.00, "Surgical": 1.65, "ICU": 2.40},
            "member_age_band": {"0-17": 0.70, "18-39": 0.85, "40-59": 1.10, "60+": 1.50},
            "product_tier": {"Bronze": 0.90, "Silver": 1.00, "Gold": 1.15},
        },
    }

# 5-line agent diff:
health_agent = Agent(
    name="Claim Severity Explainer",
    model=Gemini(id=MODEL_ID),
    tools=[load_severity_table_health, check_factor_in_table, explain_factor, generate_doc],
    instructions=SYSTEM_PROMPT_V2.replace("Priya Nair", "Dr Ananya Iyer")
                                  .replace("ABC General Insurance", "ABC Health")
                                  .replace("rating", "severity"),
)
print(f"Health agent ready: {health_agent.name}")


Health agent ready: Claim Severity Explainer


### ABC Life — Mortality Assumption Documenter (5-line diff)


In [75]:
# Life swap: load the mortality assumption set instead.

def load_mortality_assumptions_life() -> dict:
    """ABC Life term mortality assumptions, by issue age band and smoker status."""
    return {
        "base_qx_per_1000": 1.0,  # at age 30, non-smoker
        "factors": {
            "issue_age_band": {"18-29": 0.85, "30-39": 1.00, "40-49": 1.55, "50-65": 2.40},
            "smoker_status": {"NS": 1.00, "S": 2.10, "Unknown": 1.50},
            "uw_route": {"NoMed": 1.20, "TeleMed": 1.05, "FullMed": 1.00},
        },
    }

# 5-line agent diff:
life_agent = Agent(
    name="Mortality Assumption Documenter",
    model=Gemini(id=MODEL_ID),
    tools=[load_mortality_assumptions_life, check_factor_in_table, explain_factor, generate_doc],
    instructions=SYSTEM_PROMPT_V2.replace("Priya Nair", "Vikram Rao")
                                  .replace("ABC General Insurance", "ABC Life")
                                  .replace("rating", "mortality assumption"),
)
print(f"Life agent ready: {life_agent.name}")


Life agent ready: Mortality Assumption Documenter


## Wrap-up

You should now be able to:

- Define a tool as a plain Python function that an Agno agent can call.
- Write a system prompt that constrains the agent to a domain and a behaviour.
- Spot the failure mode where the agent invents content the tools never gave it.
- Add a guardrail tool that closes that failure mode.
- Adapt the same agent shape to a different LOB in roughly five lines of diff.

**Where to next:** open Track 3 of the case study brief for the 2-week build.

**Companion slides:** S2.P2.02 – S2.P2.17 (S2P2_Revised.pptx) and the S1.P1.07 demo; §7 feeds S2P3_Revised.pptx.
